# 02 — QLoRA Fine-Tuning Gemma 4 E4B on Medical QA

Fine-tune Gemma 4 E4B using QLoRA (4-bit base + LoRA adapters) via **Unsloth**
on combined PubMedQA + MedQA data. Then merge adapters back into the base model
for downstream quantization.

**Run this on**: Colab Free (T4 16GB) — Unsloth loads in 4-bit, so T4 is fine for training.

In [ ]:
# Install Unsloth (handles transformers, peft, bitsandbytes deps)
!pip install -q unsloth datasets trl

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 1. Load Model in 4-bit via Unsloth

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "google/gemma-4-E4B"
MAX_SEQ_LENGTH = 1024

print(f"Loading {MODEL_NAME} in 4-bit via Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,          # QLoRA: 4-bit base
    dtype=torch.bfloat16,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 2. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                          # LoRA rank
    lora_alpha=32,                 # Scaling factor (2x rank)
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth-optimized, saves ~30% VRAM
)

model.print_trainable_parameters()
# Should show ~0.5-1% trainable params

## 3. Load Training Data

In [ ]:
from datasets import load_dataset, concatenate_datasets

# Gemma 4 prompt templates
PUBMEDQA_TEMPLATE = """<|turn>user
Context: {context}

Question: {question}
Answer with yes, no, or maybe and explain your reasoning.<turn|>
<|turn>model
{answer}<turn|>"""

MEDQA_TEMPLATE = """<|turn>user
{question}<turn|>
<|turn>model
{answer}<turn|>"""


def format_pubmedqa(example):
    # Standard pubmed_qa uses nested context dict with lowercase keys
    ctx_data = example.get("context", {})
    contexts = ctx_data.get("contexts", [])
    context_str = " ".join(contexts) if isinstance(contexts, list) else str(contexts)

    long_answer = example.get("long_answer", "")
    final_decision = example.get("final_decision", "")
    answer = f"{final_decision}. {long_answer}" if long_answer else final_decision
    return {"text": PUBMEDQA_TEMPLATE.format(context=context_str, question=example["question"], answer=answer)}


def format_medqa(example):
    question = example["question"]
    options = example.get("options", {})
    if isinstance(options, dict):
        opts_str = "\n".join(f"  {k}) {v}" for k, v in sorted(options.items()))
        question = f"{question}\n{opts_str}"
    answer_idx = example.get("answer_idx", "")
    answer = example.get("answer", "")
    full_answer = f"The answer is {answer_idx}) {answer}" if answer_idx else answer
    return {"text": MEDQA_TEMPLATE.format(question=question, answer=full_answer)}


# Load datasets
pubmedqa = load_dataset("pubmed_qa", "pqa_labeled")  # Only has 'train' (1000 examples)
medqa = load_dataset("GBaker/MedQA-USMLE-4-options")  # Has 'train' and 'test'

pubmedqa_fmt = pubmedqa.map(format_pubmedqa, remove_columns=pubmedqa["train"].column_names)
medqa_fmt = medqa.map(format_medqa, remove_columns=medqa["train"].column_names)

# Combine: PubMedQA train + MedQA train for training
# Use MedQA test as validation (PubMedQA has no val split)
train_dataset = concatenate_datasets([pubmedqa_fmt["train"], medqa_fmt["train"]])
val_dataset = medqa_fmt["test"]

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

## 4. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "gemma4-e4b-medical-qlora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,       # T4-safe (reduce to 1 if OOM)
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,       # Effective batch = 16
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    save_total_limit=2,
    bf16=True,                           # Gemma 4 native dtype
    report_to="none",                    # Set to "wandb" if you want logging
    optim="adamw_8bit",
    max_grad_norm=0.3,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Save the LoRA adapter
trainer.save_model(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

# Log training metrics
train_log = trainer.state.log_history
print(f"\nFinal train loss: {train_log[-2].get('loss', 'N/A')}")
print(f"Final eval loss:  {train_log[-1].get('eval_loss', 'N/A')}")

## 5. Merge LoRA into Base Model

Merge the adapter weights back into the full BF16 model.
This merged model is what we'll quantize in notebook 03.

In [ ]:
# Free memory from training
del model, trainer
torch.cuda.empty_cache()

MERGED_DIR = "gemma4-e4b-medical-merged"

# Reload and merge via Unsloth
print("Loading adapter model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

print("Merging LoRA weights and saving as BF16...")
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)

print(f"Done! Merged BF16 model saved to {MERGED_DIR}/")
print(f"This is what we'll quantize in notebook 03.")

In [ ]:
# Quick sanity check — load merged model in 4-bit (T4-safe) and test generation
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

del model
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR, quantization_config=bnb_config, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)

prompt = "<|turn>user\nWhat are the common symptoms of Type 2 diabetes?<turn|>\n<|turn>model\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# Evaluate the fine-tuned model (loaded in 4-bit from cell above)
# On T4 we can't load the merged BF16 model at full precision, so we eval in 4-bit.
# The perplexity/accuracy numbers reflect the fine-tuned weights (just quantized for inference).
import time
import pandas as pd
from tqdm import tqdm

device = "cuda"
ft_results = {"model": "gemma4-e4b-medical-finetuned"}
BATCH_SIZE = 2  # Safe for T4 16GB

# Perplexity - WikiText-2
print("[1/4] WikiText-2 perplexity...")
wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
wiki_texts = [t for t in wiki["text"] if len(t.strip()) > 50][:512]
model.eval()
total_loss, total_tokens = 0.0, 0
for i in tqdm(range(0, len(wiki_texts), BATCH_SIZE)):
    batch = wiki_texts[i:i+BATCH_SIZE]
    enc = tokenizer(batch, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        out = model(**enc, labels=enc["input_ids"])
    n = enc["attention_mask"].sum().item()
    total_loss += out.loss.item() * n
    total_tokens += n
ft_results["perplexity_wikitext"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
print(f"  -> {ft_results['perplexity_wikitext']:.2f}")

# Perplexity - Medical
print("[2/4] Medical text perplexity...")
pqa_eval = load_dataset("pubmed_qa", "pqa_labeled", split="train")
med_texts = []
for ex in pqa_eval:
    ctx_data = ex.get("context", {})
    contexts = ctx_data.get("contexts", [])
    ctx = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
    if len(ctx.strip()) > 50:
        med_texts.append(ctx)
    if len(med_texts) >= 512:
        break
total_loss, total_tokens = 0.0, 0
for i in tqdm(range(0, len(med_texts), BATCH_SIZE)):
    batch = med_texts[i:i+BATCH_SIZE]
    enc = tokenizer(batch, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        out = model(**enc, labels=enc["input_ids"])
    n = enc["attention_mask"].sum().item()
    total_loss += out.loss.item() * n
    total_tokens += n
ft_results["perplexity_medical"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
print(f"  -> {ft_results['perplexity_medical']:.2f}")

# PubMedQA accuracy
print("[3/4] PubMedQA accuracy...")
correct, total = 0, 0
for ex in tqdm(pqa_eval, total=min(len(pqa_eval), 500)):
    if total >= 500:
        break
    ctx_data = ex.get("context", {})
    contexts = ctx_data.get("contexts", [])
    context = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
    prompt = (
        f"<|turn>user\nContext: {context}\n\n"
        f"Question: {ex['question']}\n"
        f"Answer with exactly one word: yes, no, or maybe.<turn|>\n"
        f"<|turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    pred = resp.split()[0].strip(".,!?;:") if resp.split() else ""
    gold = ex["final_decision"].lower().strip()
    if pred in ("yes", "no", "maybe") and pred == gold:
        correct += 1
    total += 1
ft_results["pubmedqa_accuracy"] = correct / total if total else 0
print(f"  -> {ft_results['pubmedqa_accuracy']:.4f} ({correct}/{total})")

# Speed & memory
print("[4/4] Inference speed & memory...")
prompt = "<|turn>user\nWhat is diabetes?<turn|>\n<|turn>model\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    model.generate(**inputs, max_new_tokens=50, do_sample=False)
total_tok, total_t = 0, 0.0
for _ in range(20):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    torch.cuda.synchronize()
    total_tok += out.shape[1] - inputs["input_ids"].shape[1]
    total_t += time.perf_counter() - t0
ft_results["tokens_per_sec"] = total_tok / total_t
ft_results["peak_vram_gb"] = torch.cuda.max_memory_allocated() / (1024**3)
print(f"  -> {ft_results['tokens_per_sec']:.1f} tok/s, {ft_results['peak_vram_gb']:.2f} GB VRAM")

# Save
import os
os.makedirs("results/tables", exist_ok=True)
df_new = pd.DataFrame([ft_results])
if os.path.exists("results/tables/all_results.csv"):
    df = pd.read_csv("results/tables/all_results.csv")
    df = df[df["model"] != ft_results["model"]]
    df = pd.concat([df, df_new], ignore_index=True)
else:
    df = df_new
df.to_csv("results/tables/all_results.csv", index=False)
print("\nFine-tuned results saved.")
df